In [ ]:
import random
import numpy as np
import time
import numba
import math
from numpy.random import Generator, PCG64DXSM, SeedSequence
import multiprocessing as mp
from structure_ORIGINAL_FNs_singlesideBASEreinLEARNING2_2026 import play_sequence

np.set_printoptions(suppress=True)

In [2]:
# use this cell for setting initial perameters

# rng = np.random.default_rng()

sides = 2
pairs = np.asarray([[0, 1], [1, 0], [1, 2], [2, 1], [2, 3], [3, 2], [3, 4], [4, 3]])
testpairs = np.asarray([[1, 3], [3, 1]])
nonadjpairs = np.asarray([[0, 2], [2, 0], [0, 3], [3, 0], [0, 4], [4, 0], [1, 4], [4, 1], [2, 4], [4, 2]])
allpairs = np.concatenate((pairs, nonadjpairs, testpairs))

plen = len(pairs)
alen = len(allpairs)

terms = 5
maxvalue = 10
startstop = 2
noise = 0.
annealing = 0.

timesteps = 10**8
runs = 1000

rein1 = 4
rein2 = 4
punish1 = -11
punish2 = -11
inertia = 1

nsteps = 100
blocklength = nsteps*1


In [3]:
# use this cell for running the code

#measuring how long code takes to run
start = time.perf_counter()



#nprocs = mp.cpu_count()-1
pool = mp.Pool(processes=3)

sg = SeedSequence()
rgs = [Generator(PCG64DXSM(s)) for s in sg.spawn(runs)]




mpseq_final = pool.starmap(play_sequence, [(n, rgs[n], rein1, punish1, rein2, punish2, timesteps, nsteps, sides, pairs, testpairs, nonadjpairs, allpairs, plen, alen, terms, maxvalue, startstop, noise, annealing, runs, inertia, blocklength) for n in range(runs)])

final_sigweights = []
final_cumsuc = []
final_cumsucadd = []
final_testcumsucadd = []
final_recweights = []

for res_idx in range(0, len(mpseq_final)):
    
    final_sigweights.append(mpseq_final[res_idx][0])
    final_cumsuc.append(mpseq_final[res_idx][1])
    final_cumsucadd.append(mpseq_final[res_idx][2])
    final_testcumsucadd.append(mpseq_final[res_idx][3])
    final_recweights.append(mpseq_final[res_idx][4])

final_sigweights = np.asarray(final_sigweights)
final_cumsuc = np.asarray(final_cumsuc)
final_cumsucadd = np.asarray(final_cumsucadd)
final_testcumsucadd = np.asarray(final_testcumsucadd)
final_recweights = np.asarray(final_recweights)



    
print("average cumsuc = ")
print(np.average(final_cumsuc)/runs)
print(" ")

print("average final cumsucadd = ")
print(np.average(final_cumsucadd)/(100))
print(" ")

print("average test cumsum = ")
print(np.average(final_testcumsucadd)/(100))
print(" ")

finish = time.perf_counter()
print(f'Finished in {round(finish-start,0)/60} minutes')

average cumsuc = 
99886.647807
 
average final cumsucadd = 
0.99908
 
average test cumsum = 
0.60202
 
Finished in 556.9 minutes


In [4]:

np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_sigweights', final_sigweights)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_recweights', final_recweights)
np.save('structure_noiseAnn_dsktp_2s_BASElearning-MV10_testcumsucadd', final_testcumsucadd)



In [8]:
cutoff = 90
final_test_count = 0
for cumsum in final_testcumsucadd:
    if cumsum > cutoff:
        final_test_count += 1
print(final_test_count)

255


In [6]:
@numba.njit
def duplicates(dbins):
    dup = 0
    for iii in range(len(dbins)):
        for jjj in range(len(dbins)):
            if iii != jjj:
                if (dbins[iii][0] == dbins[jjj][0]) & (dbins[iii][1] == dbins[jjj][1]):
                    dup = 1
    return dup

# function to check how many runs had duplicate bins
@numba.njit
def runs_dup_bins(runs, fsigweights, t):
    dup_count = 0
    for i0 in range(runs):
        sw = fsigweights[i0].copy()
        swl = sw.argmax(axis=3)[0, 0]
        swr = sw.argmax(axis=3)[0, 1]
        bns = np.zeros(((2*(t-1)), 2), dtype = numba.int64)
        for idx000 in range(t-1):
            bns[idx000][0] = swl[idx000] 
            bns[idx000][1] = swr[idx000+1]
        for idx001 in range(t-1):
            bns[t-1+idx001][0] = swl[idx001+1] 
            bns[t-1+idx001][1] = swr[idx001]
        dup_count += duplicates(bns)
    return dup_count

In [7]:
runs_dup_bins(runs, final_sigweights, terms)

149